# ED Journal

> Processing journal events

In [ ]:
#| default_exp edjournal

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import time
import hashlib
import json
import logging
import datetime
import ntpath
from typing import Iterator
from functools import reduce


In [ ]:
from confproxy.core import init_console_logging
from edcompanion.core import configuration


In [ ]:
configuration["FOLDERS"]["ed_journals"]

'/home/fenke/Saved Games/Frontier Developments/Elite Dangerous'

In [ ]:
init_console_logging(__name__)

2025-12-23T20:05:46+0100 INFO	8034	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| export
syslog = logging.getLogger(__name__)


In [ ]:
? os.listdir

Signature:  os.listdir(path=None)
Docstring:
Return a list containing the names of the files in the directory.

path can be specified as either str, bytes, or a path-like object.  If path is bytes,
  the filenames returned will also be bytes; in all other circumstances
  the filenames returned will be str.
If path is None, uses the path='.'.
On some platforms, path may also be specified as an open file descriptor;\
  the file descriptor must refer to a directory.
  If this functionality is unavailable, using it raises NotImplementedError.

The list is in arbitrary order.  It does not include the special
entries '.' and '..' even if they are present in the directory.
Type:      builtin_function_or_method

## Journal files

### List journal files

In [ ]:
#| export

def list_journals_unsorted(journalpath:str)->Iterator[str]:
    'Generator for journal file paths'
    for jn in (os.path.join(journalpath, f) for f in os.listdir(journalpath) if 'Journal' in f.split('.')[0] and '.log' in f):
        yield jn


In [ ]:
print(list(list_journals_unsorted(configuration["FOLDERS"]["ed_journals"])))

['/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-27T191220.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-25T222309.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-25T212333.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T160055.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T151651.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T132129.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-23T153615.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T215106.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T113412.01.log', '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T113234.01.log', '/home/fenke/Saved 

In [ ]:
#| export

def list_journals_sorted(journalpath:str)->list[str]:
    'Li'
    return sorted(
            [os.path.join(journalpath, f) for f in os.listdir(journalpath) if 'Journal' in f.split('.')[0] and '.log' in f],
            key=lambda f:f.replace('-', '').replace('Journal.20', 'Journal.').replace('T','')
        )


In [ ]:
#| export

def list_journals(journalpath:str, sorted:bool=True)->Iterator[str]:
    if sorted:
        for jn in list_journals_sorted(journalpath):
            yield jn
    else:
        yield from list_journals_unsorted(journalpath)


In [ ]:
list(list_journals(configuration["FOLDERS"]["ed_journal_archive"], False))

['/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-27T191220.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-25T222309.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-25T212333.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T160055.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T151651.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T132129.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-23T153615.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T215106.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T113412.01.log',
 '/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T113234.01.log',
 '/home/fe

### List journal events

In [ ]:
#| export

def read_journal(
    journal:str,    # path to journal file
    tail=True       # finish on end of input or wait for new events
)->Iterator[dict]:
    """
        Returns a generator of journal events
        journal: path to journal
        tail:    finish on end of input or wait for new events (finishes on 'Shutdown' event)
    """

    syslog = logging.getLogger(f"root.{__name__}")
    last_timestamp = None
    
    try:
        syslog.info(f"\nReading journal: {ntpath.basename(journal)}")

        with open(journal, encoding="utf-8") as journalfile:
            syslog.debug(f"Opening journal {journal}")
            while True: # not shutdown_seen:
                line = journalfile.readline()
                if not line:
                    if tail:
                        time.sleep(0.3)
                        continue
                    else:
                        break

                if len(line) < 5:
                    continue

                try:
                    event = json.loads(line)
                    if not last_timestamp:
                        event['logfile'] = ntpath.basename(journal)
                        
                    last_timestamp = event.get('timestamp', last_timestamp)

                except json.decoder.JSONDecodeError as JX:
                    syslog.exception("Exception: %s", JX, exc_info=True, stack_info=True)
                    return

                yield event
        
                if event.get('event', '') == 'Shutdown':
                    syslog.info(f"SHUTDOWN {event.get('timestamp'):22} {ntpath.basename(journal)}")
                    break



    except KeyboardInterrupt:
        syslog.info("Keyboard Interrupt")
        yield dict(
            event='KeyboardInterrupt',
            timestamp=last_timestamp,
            filename=f"{ntpath.basename(journal)}"
        )

    finally:
        syslog.debug(f"Done reading journal: {journal}")
        yield dict(
            event='JournalFinished',
            timestamp=last_timestamp,
            filename=f"{ntpath.basename(journal)}"
        )



In [ ]:
journalfile = list_journals_sorted(configuration["FOLDERS"]["ed_journal_archive"])[-1]
print(journalfile)
print(json.dumps(list(read_journal(journalfile)), indent=2))

/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-27T191220.01.log
[
  {
    "timestamp": "2023-06-27T17:12:14Z",
    "event": "Fileheader",
    "part": 1,
    "language": "English/UK",
    "Odyssey": true,
    "gameversion": "4.0.0.1502",
    "build": "r294054/r0 ",
    "logfile": "Journal.2023-06-27T191220.01.log"
  },
  {
    "timestamp": "2023-06-27T17:13:14Z",
    "event": "Commander",
    "FID": "F9569960",
    "Name": "immerlicht"
  },
  {
    "timestamp": "2023-06-27T17:13:14Z",
    "event": "Materials",
    "Raw": [
      {
        "Name": "zinc",
        "Count": 134
      },
      {
        "Name": "sulphur",
        "Count": 110
      },
      {
        "Name": "niobium",
        "Count": 14
      },
      {
        "Name": "phosphorus",
        "Count": 119
      },
      {
        "Name": "iron",
        "Count": 113
      },
      {
        "Name": "carbon",
        "Count": 116
      },
      {
        "Name": "vanadium",
        "Count": 3
 

In [ ]:
backlog = 8

logfiles = list_journals_sorted(configuration["FOLDERS"]["ed_journal_archive"])

backlog = min(backlog, len(logfiles)-1)

print(json.dumps(logfiles[-(1+backlog):], indent=4))

[
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T113412.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-22T215106.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-23T153615.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T132129.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T151651.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-24T160055.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-25T212333.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-25T222309.01.log",
    "/home/fenke/Saved Games/Frontier Developments/Elite Dangerous/Journal.2023-06-27T191220.01.log"
]


In [ ]:
backlog = '2024-09-30' # read from journal matching this pattern
reduce(
                lambda t, j: t if not t and backlog not in j else t + [j],
                logfiles, []
            )

[]

In [ ]:
#| export

def track_journals(journalpath:str, backlog:int|str=0):
    '''Iterable for Journal events spanning multiple log files'''

    try:

        #logfiles = sorted(glob.glob(os.path.join(journalpath, journalglob)))
        logfiles = list_journals_sorted(journalpath)

        if isinstance(backlog, int):
            backlog = min(backlog, len(logfiles)-1)
            syslog.info(f"Reading journals, backlog = {backlog}")
            for f in logfiles[-(1+backlog):]:
                yield from read_journal(f)

        elif isinstance(backlog, str):
            syslog.info(f"Reading journals, backlog = {backlog}")
            for f in reduce(
                lambda t, j: t if not t and backlog not in j else t + [j],
                logfiles, []
            ):
                yield from read_journal(f)


    except KeyboardInterrupt as kbi:
        syslog.info(f"Keyboard Interrupt {kbi.info()}")
        



In [ ]:
configuration["FOLDERS"]["ed_journal_archive"]

'/home/fenke/Saved Games/Frontier Developments/Elite Dangerous'

In [ ]:
for j in track_journals(configuration["FOLDERS"]["ed_journal_archive"], backlog=0):
    print(json.dumps(j, indent=4))

2025-12-23T20:05:46+0100 INFO	8034	__main__	752592036.py	track_journals	13	Reading journals, backlog = 0


{
    "timestamp": "2023-06-27T17:12:14Z",
    "event": "Fileheader",
    "part": 1,
    "language": "English/UK",
    "Odyssey": true,
    "gameversion": "4.0.0.1502",
    "build": "r294054/r0 ",
    "logfile": "Journal.2023-06-27T191220.01.log"
}
{
    "timestamp": "2023-06-27T17:13:14Z",
    "event": "Commander",
    "FID": "F9569960",
    "Name": "immerlicht"
}
{
    "timestamp": "2023-06-27T17:13:14Z",
    "event": "Materials",
    "Raw": [
        {
            "Name": "zinc",
            "Count": 134
        },
        {
            "Name": "sulphur",
            "Count": 110
        },
        {
            "Name": "niobium",
            "Count": 14
        },
        {
            "Name": "phosphorus",
            "Count": 119
        },
        {
            "Name": "iron",
            "Count": 113
        },
        {
            "Name": "carbon",
            "Count": 116
        },
        {
            "Name": "vanadium",
            "Count": 3
        },
        {
       

In [ ]:
event_types = set()
for j in track_journals(configuration["FOLDERS"]["ed_journals"], backlog=3):
    event_types.add(j['event'])

print(event_types)


2025-12-23T20:05:46+0100 INFO	8034	__main__	752592036.py	track_journals	13	Reading journals, backlog = 3


{'StartJump', 'SupercruiseDestinationDrop', 'Commander', 'Location', 'RefuelAll', 'Outfitting', 'Rank', 'StoredModules', 'ReceiveText', 'Progress', 'ModuleBuy', 'NavRouteClear', 'ShipyardSwap', 'ShipLocker', 'Cargo', 'MultiSellExplorationData', 'TechnologyBroker', 'FetchRemoteModule', 'NavRoute', 'Docked', 'Reputation', 'ModuleSell', 'FSSSignalDiscovered', 'JournalFinished', 'Shipyard', 'Scan', 'Music', 'Fileheader', 'FuelScoop', 'DockingRequested', 'EngineerProgress', 'ModuleInfo', 'DockingGranted', 'Missions', 'FSDTarget', 'FSDJump', 'Materials', 'ShipyardBuy', 'Undocked', 'SupercruiseExit', 'ModuleRetrieve', 'MaterialTrade', 'ShipyardNew', 'FSSDiscoveryScan', 'Statistics', 'Loadout', 'StoredShips', 'LoadGame', 'Shutdown', 'ModuleStore', 'ShieldState'}


### Utilities

In [ ]:
hash_obj = hashlib.md5("salted".encode("utf-8"))
hash_obj, hash_obj.hexdigest()

(<md5 _hashlib.HASH object @ 0x7f7ea42d4e90>,
 '51b6e225b30508aede8d7763cbc83a1a')

In [ ]:
c1 = hash_obj.copy()
c1.update("text".encode("utf-8"))
c1, c1.hexdigest()

(<md5 _hashlib.HASH object @ 0x7f7ea42d4f70>,
 '5e0f0a3e858cc040577e7d8c0ef100cd')

In [ ]:
c2 = hash_obj.copy().update("text".encode("utf-8"))
c2

In [ ]:
hash_obj, hash_obj.hexdigest()

(<md5 _hashlib.HASH object @ 0x7f7ea42d4e90>,
 '51b6e225b30508aede8d7763cbc83a1a')

In [ ]:
hashlib.md5(json.dumps({
    "timestamp": "2023-06-27T17:13:14Z",
    "event": "Commander",
    "FID": "F9569960",
    "Name": "immerlicht"
}).encode("utf-8")).hexdigest()

'd6a1e883fd111de0cfc389ee22fb031a'

In [ ]:
h = hashlib.md5("salt".encode("utf-8"))



In [ ]:
h, h.hexdigest()

(<md5 _hashlib.HASH object @ 0x7f7ea42d7350>,
 'ceb20772e0c9d240c75eb26b0e37abee')

In [ ]:
h.update("text".encode("utf-8"))

In [ ]:
h, h.hexdigest()

(<md5 _hashlib.HASH object @ 0x7f7ea42d7350>,
 '49625075c45179ce5a7a601c8ea8858a')

In [ ]:
#| export


def salted_event_hasher(salt:str):

    def _hash(event:dict):
        h = hashlib.md5(salt.encode("utf-8"))
        h.update(
            json.dumps(event).encode("utf-8")
        
        )
        return h
    return _hash

    

In [ ]:
hash_from_event = salted_event_hasher("SALT is a movie with Angelina Jolie")

hash_from_event

<function __main__.salted_event_hasher.<locals>._hash(event: dict)>

In [ ]:
hash_from_event({
    "timestamp": "2023-06-27T17:13:14Z",
    "event": "Commander",
    "FID": "F9569960",
    "Name": "immerlicht"
}).hexdigest()

'16bc7a3592e3deddffb6ef595171939d'

## Journaling DB

### Files & player

## Processing event


### Systems, FSD & location


### Ship, modules & materials

### Scanning & signals

In [ ]:
import nbdev; nbdev.nbdev_export()